# Building a first Classification NN

Use the Ames Mutagenicity dataset (from assignment 1A) and build a binary classifier NN. Play with the model parameters. 

For comparison of the NN model performance, consider the performance of other (baseline) classifier models (assignment 1A):
- KNN: Test-Accuracy 0.79, Test-ROC-AUC 0.86
- Decision Tree: Test-Accuracy 0.78, Test-ROC-AUC 0.77
- Random Forest: Test-Accuracy 0.83, Test-ROC-AUC 0.90
- Gradient Boosting: Test-Accuracy 0.77, Test-ROC-AUC 0.85


#### Tasks:
1) Load the dataset `ames_data.csv`. The dataset does not contain any duplicates or NaNs
2) Feature engineering: Calculate various fingerprints from the SMILES strings via mol objects using RDKit(snippet provided for Morgan FPs and MACCS keys)
3) Create feature matrix and target vector. Choose first the MorganFP (Later repeat the process for other fingerprint types). Convert the training and test sets into pytorch tensors.
4) Build your NN (see below for more info)
5) Train your model on the Morgan Fingerprints (and repeat later for other FP types)
6) Evaluate your model's performance and compare to other classifier models
7) Save the model / current state.
8) Respond to the discussion points


#### Note:
The aim of this exercise is to gain a bit of practice in building a simple NN and to see how different parameters and feature engineering influence the model. Maximum accuracy is not the target. 

There is no need to venture too far into the details or more advanced approaches just yet (e.g. batched training would be complete overkill for this assignment - we will discuss that in the next sessions)

0) Import dependencies and datasets

In [2]:
# complete imports if needed for your solution
import pandas as pd
import numpy as np

from rdkit import Chem
from rdkit.Chem import rdFingerprintGenerator
from rdkit.Chem import MACCSkeys

from sklearn.model_selection import train_test_split

import torch
import torch.nn as nn
import torch.optim as optim

1) Load and investigate the data

In [3]:
df = pd.read_csv("ames_data.csv")
df.head()

,drug_id,smiles,mutagenicity
0,Drug 0,O=[N+]([O-])c1ccc2ccc3ccc([N+](=O)[O-])c4c5ccc...,1
1,Drug 1,O=[N+]([O-])c1c2c(c3ccc4cccc5ccc1c3c45)CCCC2,1
2,Drug 2,O=c1c2ccccc2c(=O)c2c1ccc1c2[nH]c2c3c(=O)c4cccc...,0
3,Drug 3,[N-]=[N+]=CC(=O)NCC(=O)NN,1
4,Drug 4,[N-]=[N+]=C1C=NC(=O)NC1=O,1


2) Generate different fingerprints (try at least one additional FP type as provided in RDKit and use two different fpSizes on one of them) - all of them will be saved in new columns in the Dataframe.

In [5]:
def smiles_to_mol(smiles):
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        return None
    return mol

def morganfp(mol):
    fp = rdFingerprintGenerator.GetMorganGenerator(radius=2, fpSize=2048).GetFingerprint(mol)
    return np.array(fp)

def maccskeys(mol):
    fp = MACCSkeys.GenMACCSKeys(mol)
    return np.array(fp)

def topological_torsions(mol):
    fp = rdFingerprintGenerator.GetTopologicalTorsionGenerator(fpSize=2048).GetFingerprint(mol)
    return np.array(fp) 

def morganfp_1024(mol):
    fp = rdFingerprintGenerator.GetMorganGenerator(radius=2, fpSize=1024).GetFingerprint(mol)
    return np.array(fp)

def topological_torsions_1024(mol):
    fp = rdFingerprintGenerator.GetTopologicalTorsionGenerator(fpSize=1024).GetFingerprint(mol)
    return np.array(fp) 

fpgens = {
    "MorganFP": morganfp,
    "MorganFP_1024": morganfp_1024,
    "MACCSkeys": maccskeys,
    "TopologicalTorsions": topological_torsions,
    "TopologicalTorsions_1024": topological_torsions_1024
}

df["mol"] = df["smiles"].apply(smiles_to_mol)

for name, fpgen in fpgens.items():
    df[name] = df["mol"].apply(fpgen)
df.head()

,drug_id,smiles,mutagenicity,mol,MorganFP,MorganFP_1024,MACCSkeys,TopologicalTorsions,TopologicalTorsions_1024
0,Drug 0,O=[N+]([O-])c1ccc2ccc3ccc([N+](=O)[O-])c4c5ccc...,1,<rdkit.Chem.rdchem.Mol object at 0x000001A7961...,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, ...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, ..."
1,Drug 1,O=[N+]([O-])c1c2c(c3ccc4cccc5ccc1c3c45)CCCC2,1,<rdkit.Chem.rdchem.Mol object at 0x000001A7961...,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ..."
2,Drug 2,O=c1c2ccccc2c(=O)c2c1ccc1c2[nH]c2c3c(=O)c4cccc...,0,<rdkit.Chem.rdchem.Mol object at 0x000001A7961...,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 1, 0, ...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 1, 0, ..."
3,Drug 3,[N-]=[N+]=CC(=O)NCC(=O)NN,1,<rdkit.Chem.rdchem.Mol object at 0x000001A7961...,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ..."
4,Drug 4,[N-]=[N+]=C1C=NC(=O)NC1=O,1,<rdkit.Chem.rdchem.Mol object at 0x000001A7961...,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, ...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ..."


3. Create feature matrix and target vector. The snippet below converts the data into numpy arrays. Start with the Morgan Fingerprints (and later return here to apply your modell to different fingerprint types - not all of the fingerprints may have the same length, so you may have to adapt the width of your layers).

Do a train startified test split and convert into pytorch tensors.

In [6]:
X = np.stack(df["MorganFP"].values) # joins multiple arrays along a new axis - builds a proper array from the line-by-line arrays in the df.
y = df["mutagenicity"].to_numpy()

X = torch.tensor(X, dtype=torch.float32)
y = torch.tensor(y, dtype=torch.float32).unsqueeze(1) # unsqueeze to make it a column vector

4) Build the NN - adhere to some robust standard values. Start simple and train the model on Morgan FP first.

Optimise the model parameters based on observed over-/underfitting. Experiment with different width and depth, as well as other model parameters. Explore some options to prevent overfitting, e.g. Early stopping (e.g. manually by limiting the epochs) or dropouts. 

Note: Since the input layer needs a lot of neurons (e.g. 2048 bit in the MFPs), consider shrinking the widht from layer to layer. 

Hint: If you use `BCELoss()` as loss function, combine it with a `sigmoid` activation in the last layer. If you use `BCEWithLogitsLoss()`, do not specify any activation in the forward pass (`x = self.outputlayer(x)`).

In [ ]:
class BinClassifierNN(nn.Module):
    def __init__(self):
        super(BinClassifierNN, self).__init__()
        self.fc1 = nn.Linear(2048, 128) # Input layer to hidden layer
        self.relu = nn.ReLU() # Activation function
        self.fc2 = nn.Linear(128, 1) # Hidden layer to output layer
        self.sigmoid = nn.Sigmoid() # Sigmoid activation for binary classification

    def forward(self, x):
        x = torch.relu(self.fc1(x)) # Pass through first layer and apply ReLU
        x = self.sigmoid(self.fc2(x)) # Pass through second layer and apply Sigmoid
        return x

In [25]:
# Parameters (change and add as needed)
learning_rate = 0.1 #Standard settings = 0.01
num_epochs = 1000 #Standard settings = 100

In [26]:
model = BinClassifierNN()

# Define the loss function and optimizer
criterion = nn.BCELoss()

# Use Adam optimizer, which is a popular choice for training neural networks.
optimizer = optim.SGD(model.parameters(), lr=learning_rate)

5. Train the NN. Note that you may have to squeeze the output (`outputs=models(X_train).squeeze`). This will reduce the actual output of the shape ``[N, 1]`` to ``[N]``, which is comparable to y (The final layer naturally produces a column tensor, which is not directly comparable to the 1D target tensor).

In [27]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

for epoch in range(num_epochs):
    model.train()
    optimizer.zero_grad()  # Clear gradients from the previous step
    outputs = model(X_train)  # Forward pass
    loss = criterion(outputs, y_train)  # Compute loss
    loss.backward()  # Backpropagation
    optimizer.step()  # Update weights

    if (epoch+1) % 10 == 0:  # Print loss every 10 epochs
        print(f'Epoch [{epoch+1}/{num_epochs}], Loss: {loss.item():.4f}')

Epoch [10/1000], Loss: 0.6844
Epoch [20/1000], Loss: 0.6750
Epoch [30/1000], Loss: 0.6663
Epoch [40/1000], Loss: 0.6577
Epoch [50/1000], Loss: 0.6487
Epoch [60/1000], Loss: 0.6393
Epoch [70/1000], Loss: 0.6297
Epoch [80/1000], Loss: 0.6202
Epoch [90/1000], Loss: 0.6110
Epoch [100/1000], Loss: 0.6021
Epoch [110/1000], Loss: 0.5936
Epoch [120/1000], Loss: 0.5856
Epoch [130/1000], Loss: 0.5779
Epoch [140/1000], Loss: 0.5705
Epoch [150/1000], Loss: 0.5634
Epoch [160/1000], Loss: 0.5565
Epoch [170/1000], Loss: 0.5498
Epoch [180/1000], Loss: 0.5434
Epoch [190/1000], Loss: 0.5371
Epoch [200/1000], Loss: 0.5311
Epoch [210/1000], Loss: 0.5252
Epoch [220/1000], Loss: 0.5196
Epoch [230/1000], Loss: 0.5142
Epoch [240/1000], Loss: 0.5090
Epoch [250/1000], Loss: 0.5039
Epoch [260/1000], Loss: 0.4991
Epoch [270/1000], Loss: 0.4944
Epoch [280/1000], Loss: 0.4898
Epoch [290/1000], Loss: 0.4855
Epoch [300/1000], Loss: 0.4812
Epoch [310/1000], Loss: 0.4772
Epoch [320/1000], Loss: 0.4732
Epoch [330/1000],

6) Evaluate the model. As a first metric, you can use the same loss function to evaluate the model on the test set. For better comparison with methods tested in the assignment 1A (results vide supra), use metrics from scikit-learn (e.g. the accuracy or ROC-AUC score).

Hint: for your prediction you may have to use `squeeze` again to match the target vector in the test set (e.g. ``y_pred = model(X_test).squeeze()``)

In [29]:
y_train_pred = model(X_train)
y_test_pred = model(X_test)

In [30]:
print("Training Accuracy:", ((y_train_pred > 0.5) == y_train).float().mean().item())
print("Testing Accuracy:", ((y_test_pred > 0.5) == y_test).float().mean().item())

Training Accuracy: 0.8684300780296326
Testing Accuracy: 0.8125


7) Research how you can save the model / current state for later reuse. What are different options here? How can it be loaded again?

In [31]:
torch.save(model.state_dict(), "bin_classifier_model.pth")

In [32]:
model = BinClassifierNN()
criterion = nn.BCELoss() # Binary Cross Entropy Loss for binary classification

In [41]:
#Different Fingerprints
X = np.stack(df["TopologicalTorsions"].values) 
y = df["mutagenicity"].to_numpy()

X = torch.tensor(X, dtype=torch.float32)
y = torch.tensor(y, dtype=torch.float32).unsqueeze(1) # unsqueeze to make it a column vector

class BinClassifierNN(nn.Module):
    def __init__(self):
        super(BinClassifierNN, self).__init__()
        self.fc1 = nn.Linear(1024, 4096) # Input layer to hidden layer
        self.relu = nn.ReLU() # Activation function
        self.fc2 = nn.Linear(4096, 4096) # Hidden layer to output layer
        self.sigmoid = nn.Sigmoid(4096,1) # Sigmoid activation for binary classification

    def forward(self, x):
        x = torch.relu(self.fc1(x)) # Pass through first layer and apply ReLU
        x = self.sigmoid(self.fc2(x)) # Pass through second layer and apply Sigmoid
        return x

learning_rate = 0.1
num_epochs = 1000

model = BinClassifierNN()
criterion = nn.BCELoss()

optimizer = optim.SGD(model.parameters(), lr=learning_rate)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

for epoch in range(num_epochs):
    model.train()
    optimizer.zero_grad()  # Clear gradients from the previous step
    outputs = model(X_train)  # Forward pass
    loss = criterion(outputs, y_train)  # Compute loss
    loss.backward()  # Backpropagation
    optimizer.step()  # Update weights

    if (epoch+1) % 10 == 0:  # Print loss every 10 epochs
        print(f'Epoch [{epoch+1}/{num_epochs}], Loss: {loss.item():.4f}')

y_train_pred = model(X_train)
y_test_pred = model(X_test)
print("Training Accuracy:", ((y_train_pred > 0.5) == y_train).float().mean().item())
print("Testing Accuracy:", ((y_test_pred > 0.5) == y_test).float().mean().item())



TypeError: Sigmoid.__init__() takes 1 positional argument but 3 were given

#### 8) Discussion points
1) How did your model compare to other simple ML classifiers (all used the Morgan FPs)? Discuss!


2) Did you observe any difference between different fingerprint types?
As only two fingerprints were tested against it each other, its hard to say something definetely. The training accuracy for both was similar, but the testing accuracy for the MorgenFP was better. 

3) Did the fingerprint size impact the model prediction? What message is to be learned from this?


4) What were some model parameters for decent performance depending on the fingerprint type? 


5) Was overfitting a problem? What approaches did you apply to limit that issue? What else would be possible


6) Consider the target "mutagenicity" in the context of molecular structure. What does noise mean here? How could you use such a predictive model in the lab? What other data-driven tools could be interesting in this QSAR context?
the noise is the parts of the fingerprints which have nothing to do with the mutagenicity of the molecule. the noise can also come from the fact that the mutagenicity is either 1 or 0, and if the molecule is a borderline case, it can be mislabled.

7) Why is exporting a full model usually not recommended?
Version problems